In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "xformers<0.0.28" peft accelerate bitsandbytes -q
!pip install gradio sentence-transformers -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 51.3 MB/s eta 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json, numpy as np, torch, gradio as gr
from pathlib import Path
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

DRIVE_ROOT  = Path("/content/drive/MyDrive/dappp")
TRAIN_PATH  = DRIVE_ROOT / "data" / "dataset_train.jsonl"
MERGED_PATH = DRIVE_ROOT / "outputs" / "merged_model"

# ── Model ─────────────────────────────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = str(MERGED_PATH),
    max_seq_length= 2048,
    dtype         = None,
    load_in_4bit  = True,
)
FastLanguageModel.for_inference(model)
print("Model spreman.")

# ── Banka ─────────────────────────────────────────────────────────────────────
def _parse_bio(content):
    text = content[5:] if content.startswith("Bio: ") else content
    return text.split("\n")[0].strip()

train_data = [json.loads(l) for l in open(TRAIN_PATH, encoding="utf-8")]
bank_icebreakers = [r["messages"][1]["content"] for r in train_data]
bank_bios_raw    = [_parse_bio(r["messages"][0]["content"]) for r in train_data]
print(f"Banka: {len(bank_icebreakers)} icebreakera")

# ── TF-IDF ────────────────────────────────────────────────────────────────────
print("TF-IDF fitovanje...")
_tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)
_tfidf.fit(bank_bios_raw + bank_icebreakers)
_bank_tfidf = _tfidf.transform(bank_icebreakers)

# ── MiniLM ────────────────────────────────────────────────────────────────────
print("MiniLM enkodiranje banke...")
_minilm    = SentenceTransformer("all-MiniLM-L6-v2")
_bank_embs = _minilm.encode(
    bank_icebreakers, convert_to_numpy=True,
    normalize_embeddings=True, batch_size=64, show_progress_bar=True,
)
print("Sve spremno.")

# ── Generacija ────────────────────────────────────────────────────────────────
def _generate(bio: str) -> str:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": f"Bio: {bio}\nWrite an icebreaker."}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            inputs, max_new_tokens=80, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# ── Demo funkcija ─────────────────────────────────────────────────────────────
def demo(bio_text: str, top_k: int):
    bio_text = bio_text.strip()
    if not bio_text:
        return "Unesi bio tekst.", "Unesi bio tekst.", "Unesi bio tekst."

    # TF-IDF
    tf_sc  = sk_cosine(_tfidf.transform([bio_text]), _bank_tfidf)[0]
    tf_top = np.argsort(tf_sc)[::-1][:top_k]
    tfidf_out = "\n\n".join(
        f"{i+1}. [score {tf_sc[idx]:.3f}]\n{bank_icebreakers[idx]}"
        for i, idx in enumerate(tf_top)
    )

    # MiniLM
    q_emb  = _minilm.encode([bio_text], convert_to_numpy=True, normalize_embeddings=True)
    ml_sc  = (q_emb @ _bank_embs.T)[0]
    ml_top = np.argsort(ml_sc)[::-1][:top_k]
    minilm_out = "\n\n".join(
        f"{i+1}. [score {ml_sc[idx]:.3f}]\n{bank_icebreakers[idx]}"
        for i, idx in enumerate(ml_top)
    )

    return tfidf_out, minilm_out, _generate(bio_text)

# ── Gradio ────────────────────────────────────────────────────────────────────
gr.Interface(
    fn=demo,
    inputs=[
        gr.Textbox(label="Bio tekst", placeholder="I love hiking and coffee shops. Dog owner.", lines=3),
        gr.Slider(minimum=1, maximum=5, value=3, step=1, label="Top-k"),
    ],
    outputs=[
        gr.Textbox(label="🔤 TF-IDF retrieval (iz banke)", lines=6),
        gr.Textbox(label="🧠 MiniLM retrieval (iz banke)", lines=6),
        gr.Textbox(label="✨ Fine-tuned Llama-3.2-3B (generacija)", lines=3),
    ],
    title="Icebreaker — retrieval vs generacija",
    description=(
        "**TF-IDF** — leksičko poklapanje, vraća top-k iz banke  \n"
        "**MiniLM** — semantička sličnost, vraća top-k iz banke  \n"
        "**Llama-3.2-3B + QLoRA** — generacija, nije ograničena na banku"
    ),
    examples=[
        ["I've got a few iguanas and a brother I'm still getting to know. I'm convinced dogs could learn to read.", 3],
        ["Early riser who loves working out. You'll find me in costumes whenever possible, cold drink in hand.", 3],
        ["Drawn to darker aesthetics, love long walks near trails. No kids, just me and the outdoors.", 3],
    ],
    allow_flagging="never",
).launch(share=True)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Učitavam merged model s Drivea (~6GB, par minuta)...
==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The tokenizer you are loading from '/content/drive/MyDrive/dappp/outputs/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/dappp/outputs/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Wil

Model spreman.
Banka: 1272 icebreakera
TF-IDF fitovanje...
MiniLM enkodiranje banke...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Sve spremno.


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c9653658de89c6335.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
